# Lab 2 — نظّف بيانات منافذ
**المدة:** 25 دقيقة  
**الهدف:** تحويل التواريخ، فحص النصوص، قياس الفقد، فحص التكرار والنطاقات والقيم المتطرفة.

> سنعمل على نسخة اسمها `df_clean` حتى تبقى البيانات الأصلية محفوظة.

In [ ]:
import io
import pandas as pd
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
df_clean = df.copy()
print(df_clean.shape)

## 1) حوّل أعمدة التاريخ
شغّل الخلية ثم تأكد أن النوع أصبح `datetime64[ns]`.

In [ ]:
for column in ["signup_date", "snapshot_date"]:
    df_clean[column] = pd.to_datetime(df_clean[column])

df_clean[["signup_date", "snapshot_date"]].dtypes

## 2) افحص الفئات النصية
إذا ظهرت المدينة نفسها بأكثر من كتابة، نوحّد المسافات وحالة الأحرف.

In [ ]:
display(df_clean["city"].value_counts())
df_clean["city_clean"] = df_clean["city"].str.strip().str.lower()
display(df_clean["city_clean"].value_counts())

## 3) عدّ القيم المفقودة
أكمل اسمي العمودين ثم شغّل الخلية.

In [ ]:
missing_rating = df_clean["avg_rating"].isna().sum()
missing_promo = df_clean["last_promo_used"].isna().sum()
print("avg_rating missing:", missing_rating)
print("last_promo_used missing:", missing_promo)

## 4) نفّذ معالجة مفهومة
الفراغ في `last_promo_used` يعني غالبًا أن العميل لم يستخدم عرضًا. لا نحول `avg_rating` إلى صفر؛ نضيف مؤشرًا ثم نعوض بالوسيط مؤقتًا للتجربة.

In [ ]:
df_clean["last_promo_used"] = df_clean["last_promo_used"].fillna("never_used")
df_clean["avg_rating_was_missing"] = df_clean["avg_rating"].isna().astype(int)
rating_median = df_clean["avg_rating"].median()
df_clean["avg_rating"] = df_clean["avg_rating"].fillna(rating_median)
print("وسيط التقييم:", round(rating_median, 2))

## 5) افحص التكرار والنطاق
التكرار والنطاق فحصان منفصلان.

In [ ]:
duplicate_rows = df_clean.duplicated().sum()
invalid_ratings = (~df_clean["avg_rating"].between(1, 5)).sum()
print("الصفوف المكررة:", duplicate_rows)
print("التقييمات خارج 1 إلى 5:", invalid_ratings)

## 6) اعرض أعلى خمس قيم للسلة ثم اتخذ قرارًا

In [ ]:
columns = ["customer_id", "orders_per_month", "avg_basket_sar", "city"]
high_rows = df_clean.nlargest(5, "avg_basket_sar")[columns]
display(high_rows)

اكتب قرارك: هل تحذف 534.83؟ لماذا؟  
................................................................................

In [ ]:
assert df_clean["last_promo_used"].isna().sum() == 0
assert df_clean["avg_rating"].isna().sum() == 0
assert df_clean["avg_rating_was_missing"].sum() == 14859
print("اجتزت نقطة التحقق في Lab 2 ✅")

## التسليم
نتائج الفقد والتكرار والنطاق، جدول أعلى خمس سلال، وقرار مكتوب عن القيمة 534.83.